<a href="https://colab.research.google.com/github/IgorAkO/Simulative_ML/blob/main/S_ML_2_2_6_Pandas_Prod_sale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML_symulative_2_2_6
## Глава 6 Практический блиц-тест

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

2.2.2


In [ ]:
sales = pd.read_csv('sales_data.csv')
prod = pd.read_csv('products_data.csv')
sale_prod = sales.merge(prod, left_on='product_id', right_on ='product_id', how='left', suffixes=('_prod', '_sale'))

### 1. В каком регионе было продано наибольшее количество продуктов категории "Смартфоны"?

In [ ]:
prod_id_smart = prod[prod['category'] == 'Смартфоны']['product_id'].tolist()
print(prod_id_smart)

[102, 107, 112, 117, 122, 127]


In [ ]:
region = sales[sales['product_id'].isin(prod_id_smart)].groupby('region').agg(quant_smart=('quantity', 'sum')).reset_index()
print(region.head(3))

   region  quant_smart
0  Восток            4
1   Запад            2
2   Север            1


### Вопрос 2
Какова разница между максимальной и минимальной средней ценой продуктов категории "Ноутбуки" между всеми производителями? Ответ округлите до целых.

In [ ]:
mean = sale_prod[sale_prod['category'] == 'Ноутбуки'].groupby('manufacturer').agg(mean=('price_per_unit', 'mean')).reset_index()
#mean = mean['mean'].ptp()
print(mean)

  manufacturer        mean
0            A  550.000000
1            C  583.333333
2            E  750.000000


### Вопрос №3
Какой покупатель совершил заказы на наибольшую сумму за все время?

In [ ]:
cust = sale_prod.groupby('customer_id').agg(total_price=('total_price','sum')).reset_index().sort_values('total_price', ascending=False)
print(cust)

    customer_id  total_price
2          1003         3550
6          1007         3500
0          1001         3150
5          1006         2850
8          1009         2650
9          1010         2050
10         1011         2050
1          1002         2000
3          1004         1900
7          1008         1750
14         1015         1350
4          1005         1200
13         1014         1200
11         1012          900
15         1016          800
17         1018          700
12         1013          550
16         1017          300


### Вопрос №4
Какова доля продаж продуктов категории "Планшеты" от общей выручки за все время (учитывая только продукты, информация о которых есть в products_data)? Ответ округлите до 2 знака после запятой. Разделитель разряда - точка.

In [ ]:
tot_pr_count = sale_prod.groupby('category').agg(total_price=('total_price','sum'))
price_count = (tot_pr_count / tot_pr_count.sum()).round(2)
print(price_count)

            total_price
category               
Наушники           0.18
Ноутбуки           0.23
Планшеты           0.25
Смартфоны          0.14
Телевизоры         0.19


### Вопрос №5
Какова абсолютная разница в средней цене за единицу продуктов категории "Смартфоны" между двумя регионами с наибольшей общей выручкой по всем категориям товаров (учитывая только продукты, информация о которых есть в products_data)? Ответ округлите до целых и возьмите по модулю.

In [ ]:
sp = sale_prod.copy()
region = sp.groupby('region')['total_price'].sum().sort_values(ascending=False)
#.reset_index().region.tolist()[:2]
mean_r = sp[(sp['region'].isin(region)) & (sp['category'] == 'Смартфоны')][['region', 'category', 'price_per_unit']]
#.groupby('price_per_unit').agg(mean_pr_unit=('price_per_unit','mean'))
# Правильный ответ: 275
print(region)
mean_r


region
Север     10200
Юг         8950
Восток     8500
Запад      4800
Name: total_price, dtype: int64


,region,category,price_per_unit


### Вопрос №6
Сколько в среднем уникальных категорий продуктов было продано по всем регионам?

In [ ]:
unique = sale_prod.groupby('region').agg(cat=('category','nunique'))
mean_unique = unique.cat.mean()
mean_unique


np.float64(5.0)

### Вопрос №7
Какой производитель представлен в наибольшем количестве различных категорий продуктов?

In [ ]:
sale_prod.groupby('region').agg(cat=('category','nunique'))

In [ ]:
manuf = sale_prod.groupby('manufacturer').agg(cat=('category','nunique'))
manuf

,cat
manufacturer,
A,2
B,3
C,3
D,3
E,4


### Вопрос №8
В каком месяце было продано больше всего продуктов категории "Планшеты" в регионе "Север"? (ответ представьте в виде номера месяца)

In [ ]:
sp['month'] = pd.to_datetime(sp['order_date']).dt.strftime('%Y-%m')
month = sp[(sp['category'] == 'Планшеты') & (sp['region'] == 'Север')].groupby('month')['quantity'].sum()
month

,quantity
month,
2022-08,1


### Вопрос №9
Какой производитель продал продукты на наибольшую сумму в регионе "Юг" за март 2023 года?

In [ ]:
manuf = sp[(sp['month']=='2023-03') & (sp['region']=='Юг')][['total_price','manufacturer']]
manuf

,total_price,manufacturer
29,700,B


### Вопрос №10
Сколько покупателей совершили заказы на продукты хотя бы трех разных категорий?

In [ ]:
cust = sp.groupby('customer_id').agg(quantity=('category','nunique')).sort_values('quantity', ascending=False)
cust[:5]

,quantity
customer_id,
1003,3
1001,2
1009,2
1006,2
1008,2


### Вопрос №11
Какова доля продаж продуктов производителя "C" от общей выручки за все время в регионе "Север"? Ответ округлите до 2 знака после запятой. В качестве разделителя разрядов используйте точку.

In [ ]:
share_sales = sp[sp['region']=='Север'].groupby('manufacturer')['total_price'].sum()
share_sales = (share_sales / share_sales.sum()).round(2)
share_sales

,total_price
manufacturer,
A,0.30
B,0.10
C,0.27
D,0.08
E,0.25


### Вопрос №12
Найдите покупателей, которые купили более чем 15% уникальных связок "Категория - Производитель". Если таких покупателей несколько - напишите их через запятую с пробелом в порядке возрастания результата.

In [ ]:
from numpy.random import normal
sp['man-cat'] = sp['manufacturer'] + '-' + sp['category']
N = sp['man-cat'].nunique()
custumers = sp.groupby('customer_id').agg(uniq=('man-cat','nunique')) *100/N
#.value_counts(normalize=True)
#['man-cat'].value_counts(normalize=True)*100

custumers[custumers['uniq'] >= 15]

,uniq
customer_id,
1001,20.0
1003,20.0


### Вопрос №13
Какое максимальное количество продуктов было продано в один заказ?

In [ ]:
quant = sp.groupby('order_id').agg(q=('quantity','sum')).sort_values('q',ascending=False)
quant[:5]

,q
order_id,
28,4
37,4
12,4
3,3
16,3


### Вопрос №14
Какой производитель имеет самую высокую среднюю цену продукта среди всех категорий?

In [ ]:
max_mprice = sp.groupby('manufacturer').agg(m_price=('price_per_unit','mean')).sort_values('m_price', ascending=False)
max_mprice

,m_price
manufacturer,
E,683.333333
B,566.666667
C,525.000000
A,491.666667
D,362.500000


### Вопрос №15
Каково общее количество уникальных производителей, чьи продукты были куплены менее чем в 3 регионах?

In [ ]:
prod_reg = sp.groupby(['manufacturer', 'product_name']).agg(n_region=('region','nunique')).reset_index()
prod_reg
manuf = sp.groupby('manufacturer').agg(n_region=('region','nunique')).reset_index()
manuf

,manufacturer,n_region
0,A,4
1,B,4
2,C,4
3,D,4
4,E,4


In [ ]:
sp[:2]

,order_id,product_id,customer_id,order_date,quantity,price_per_unit,total_price,payment_method,region,product_name,category,manufacturer,month,man-cat
0,1,101,1001,2022-02-15,2,500,1000,Карта,Север,Ноутбук HP Pavilion 15,Ноутбуки,A,2022-02,A-Ноутбуки
1,2,102,1002,2022-03-20,1,800,800,Наличные,Юг,Смартфон Samsung Galaxy S21,Смартфоны,B,2022-03,B-Смартфоны
